<a href="https://colab.research.google.com/github/pevu97/EdgeAI/blob/main/EdgeAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
shutil.rmtree(train_dir)
shutil.rmtree(valid_dir)
shutil.rmtree(test_dir)

In [16]:
import os
import shutil
import random

base_dir = '/content/data'
data_dir = './images'

train_dir = os.path.join(data_dir, 'train')
valid_dir = os.path.join(data_dir, 'valid')
test_dir = os.path.join(data_dir, 'test')

for directory in (data_dir, train_dir, valid_dir, test_dir):
    os.makedirs(directory, exist_ok=True)

# tylko obrazy
image_files = [
    fname for fname in os.listdir(base_dir)
    if fname.lower().endswith(('.png', '.jpg', '.jpeg'))
]

random.seed(42)
random.shuffle(image_files)

size = len(image_files)

train_size = int(0.7 * size)
valid_size = int(0.2 * size)
test_size = size - train_size - valid_size

train_files = image_files[:train_size]
valid_files = image_files[train_size:train_size + valid_size]
test_files = image_files[train_size + valid_size:]

for fname in train_files:
    shutil.copyfile(
        os.path.join(base_dir, fname),
        os.path.join(train_dir, fname)
    )

for fname in valid_files:
    shutil.copyfile(
        os.path.join(base_dir, fname),
        os.path.join(valid_dir, fname)
    )

for fname in test_files:
    shutil.copyfile(
        os.path.join(base_dir, fname),
        os.path.join(test_dir, fname)
    )

print('zbiór treningowy:', len(os.listdir(train_dir)))
print('zbiór walidacyjny:', len(os.listdir(valid_dir)))
print('zbiór testowy:', len(os.listdir(test_dir)))

zbiór treningowy: 16
zbiór walidacyjny: 4
zbiór testowy: 4


In [18]:
import torch
import torch.nn as nn


class ConvAutoencoder(nn.Module):
  def __init___(self):
    super().__init__()

    self.encoder = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),
        nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),
        nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
        nn.ReLU()
    )

    self.decoder = nn.Sequential(
        nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),  # 16 -> 32
        nn.ReLU(),
        nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),   # 32 -> 64
        nn.ReLU(),
        nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),   # 64 -> 128
        nn.ReLU(),
        nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),    # 128 -> 256
        nn.Sigmoid()
    )
def forward(self, x):
  x = self.encoder(x)
  x = self.decoder(x)
  return x


In [22]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms


transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

class ImageFolderAutoencoderDataset(Dataset):
  def __init__(self, folder_path, transform=None):
    self.folder_path = folder_path
    self.transform = transform
    self.image_files = [
        fname for fname in os.listdir(folder_path)
        if fname.lower().endswith(('.png', '.jpg', '.jpeg'))

    ]

  def __len__(self):
    return len(self.image_files)

  def __getitem__(self, idx):
    img_name = self.image_files[idx]
    img_path = os.path.join(self.folder_path, img_name)

    image = Image.open(img_path).convert('L')

    if self.transform:
      image = self.transform(image)

    return image

In [23]:
train_dataset = ImageFolderAutoencoderDataset(train_dir, transform=transform)
valid_dataset = ImageFolderAutoencoderDataset(valid_dir, transform=transform)
test_dataset = ImageFolderAutoencoderDataset(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)